In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [2]:
text = "my name is john doe, email is john@email.com and age is 23"

In [3]:
res = llm.invoke(f"give me only name, email and age from this text:{text}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [4]:
res

AIMessage(content=[{'type': 'text', 'text': 'Name: john doe\nEmail: john@email.com\nAge: 23', 'extras': {'signature': 'El4KXAFpFH0TS9nSCeAoDubwq+K0PQDXO9qBUgpEHSitAK2BUBX6gue9ZXbB43GBlLOpsB9oMJqAJLo+wDfnQlY57gVRtxJ1Wu8SSFusYGDkIiVA084w6Wh0Atu7hilR'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d87d-8e2d-7132-93f6-fd9fb6f73804-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 18, 'total_tokens': 50, 'input_token_details': {'cache_read': 0}})

In [5]:
if isinstance(res.content, list):
    # Extract text blocks (skips extra metadata like 'extras' or signatures)
    text_content = "".join(
        block["text"] for block in res.content if isinstance(block, dict) and "text" in block
    )
else:
    text_content = res.content

print("Agent:", text_content, "\n")

Agent: Name: john doe
Email: john@email.com
Age: 23 



#### solution is strectured Output
---
Output: Name: john doe\nEmail: john@email.com\nAge: 23 
> previous output is text, we cannot extract it.

In [10]:
from pydantic import BaseModel, Field
from typing import List

class ResponseStrectured(BaseModel):
    name:str = Field(description="Complete Name")
    email:str = Field(description="Email Address")
    age:str = Field(description="Age")

strectured_llm = llm.with_structured_output(ResponseStrectured)

In [4]:
res = strectured_llm.invoke(f"give me only name, email and age from this text:{text}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [5]:
res

ResponseStrectured(name='john doe', email='john@email.com', age='23')

In [6]:
print(res.name)
print(res.model_dump())

john doe
{'name': 'john doe', 'email': 'john@email.com', 'age': '23'}


In [14]:
class Movie(BaseModel):
    title:str = Field(description="movie title")
    year:int = Field(description="movie released year")

class AllMovies(BaseModel):
    movies:List[Movie]

# movie_llm = llm.with_structured_output(Movie) # Movie for single movie
movie_llm = llm.with_structured_output(AllMovies)

In [15]:
resMovie = movie_llm.invoke("give me 10 most popular sci-fi recent movies")

In [16]:
resMovie

AllMovies(movies=[Movie(title='Dune: Part Two', year=2024), Movie(title='Oppenheimer', year=2023), Movie(title='Spider-Man: Across the Spider-Verse', year=2023), Movie(title='Guardians of the Galaxy Vol. 3', year=2023), Movie(title='Avatar: The Way of Water', year=2022), Movie(title='Everything Everywhere All at Once', year=2022), Movie(title='Dune', year=2021), Movie(title='Spider-Man: No Way Home', year=2021), Movie(title='Tenet', year=2020), Movie(title='Blade Runner 2049', year=2017)])